# Módulo EP1 — Prompts, Control de Contexto y RAG Dual

Correspondencia directa con:
- `app/rag/prompts.py`
- `app/rag/context.py`
- `app/rag/loaders.py` y `app/rag/chunking.py`
- `app/rag/vectorstore.py`, `app/rag/retriever.py` y `app/rag/pipeline.py`


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HikariLucy/KnowledgeFlow-AgentOps.git"
REPO_BRANCH = "feat/eft-notebook"

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "app" / "main.py").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None and IN_COLAB:
    target = Path("/content/KnowledgeFlow-AgentOps")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(target / "requirements.txt")], check=True)
    repo_root = target

if repo_root is None:
    raise RuntimeError("No se encontró la raíz de KnowledgeFlow-AgentOps. Ejecuta desde el repositorio o en Google Colab.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Seguridad: modo offline sin claves de Gemini por defecto
os.environ.setdefault("OBS_EXECUTION_MODE", "offline")
os.environ.pop("GEMINI_API_KEY", None)
RUN_LIVE_GEMINI = False

print(f"Entorno listo | IN_COLAB={IN_COLAB} | repo_root={repo_root} | RUN_LIVE_GEMINI={RUN_LIVE_GEMINI}")


## 1. Inspección fiel del código fuente EP1 (sin duplicación manual)


In [ ]:
import inspect
from app.rag import prompts, context, pipeline

print("=== app/rag/prompts.py (extracto) ===")
print(inspect.getsource(prompts.format_rag_user_prompt))
print("=== app/rag/context.py :: build_rag_context ===")
print(inspect.getsource(context.build_rag_context))


## 2. Ejecución offline de indexación FAISS y recuperación dual (interna / externa / all)


In [ ]:
import hashlib
import math
import numpy as np
from app.core.config import Settings
from app.rag.embeddings import BaseEmbeddings, prepare_document_for_embedding, prepare_query_for_embedding
from app.rag.generator import BaseRAGGenerator
from app.agents.source_router import BaseSourceRouter
from app.rag.schemas import RouteDecision

class OfflineDeterministicEmbeddings(BaseEmbeddings):
    """Doble determinista offline (bag-of-words hashing) para demostrar FAISS sin llamar a Gemini."""
    def __init__(self, dimension: int = 128) -> None:
        self.dimension = dimension

    def _encode(self, text: str) -> list[float]:
        vec = np.zeros(self.dimension, dtype=np.float32)
        tokens = [t.strip(".,:;!?()[]{}").lower() for t in text.split() if len(t.strip()) > 2]
        for tok in tokens:
            idx = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16) % self.dimension
            vec[idx] += 1.0
        norm = float(np.linalg.norm(vec))
        if norm > 0:
            vec /= norm
        return vec.tolist()

    def embed_document(self, document, metadata=None) -> list[float]:
        text = document.page_content if hasattr(document, "page_content") else str(document)
        meta = document.metadata if hasattr(document, "metadata") else metadata
        return self._encode(prepare_document_for_embedding(text, meta))

    def embed_documents(self, documents) -> list[list[float]]:
        return [self.embed_document(doc) for doc in documents]

    def embed_query(self, query: str) -> list[float]:
        return self._encode(prepare_query_for_embedding(query))


class OfflineRuleSourceRouter(BaseSourceRouter):
    """Enrutador determinista offline para distinguir fuentes internas, externas o mixtas."""
    def route(self, query: str) -> RouteDecision:
        q = query.lower()
        ext_hits = any(k in q for k in ("owasp", "nist", "estándar", "estandar", "iso", "externa"))
        int_hits = any(k in q for k in ("novatech", "interna", "acceso", "contraseña", "incidente", "mfa", "sla"))
        if ext_hits and int_hits:
            return RouteDecision(source_scope="all", confidence=0.85, routing_reason="Offline rule: consulta mixta")
        if ext_hits:
            return RouteDecision(source_scope="external", confidence=0.85, routing_reason="Offline rule: estándares externos")
        if int_hits:
            return RouteDecision(source_scope="internal", confidence=0.85, routing_reason="Offline rule: políticas internas")
        return RouteDecision(source_scope="all", confidence=0.55, routing_reason="Offline rule: alcance general")


class OfflineGroundedGenerator(BaseRAGGenerator):
    """Generador determinista offline que cita fragmentos [S#] reales recuperados; NO es Gemini real."""
    def generate(self, query, context_str, source_references, source_scope="all"):
        if not source_references or not context_str.strip():
            return (
                "No encontré evidencia suficiente en las fuentes disponibles para responder esta consulta.",
                [],
                False,
            )
        citations = [ref.id for ref in source_references[:2]]
        cite_str = "".join(f"[{cid}]" for cid in citations)
        files = ", ".join(f"{ref.file_name} ({ref.source_type})" for ref in source_references[:2])
        answer = (
            f"[MODO OFFLINE DETERMINISTA — NO GEMINI REAL] "
            f"Consulta ({source_scope}) fundamentada en {files} {cite_str}."
        )
        return answer, citations, True

from app.rag.loaders import load_knowledge_base
from app.rag.chunking import split_documents
from app.rag.vectorstore import VectorStore
from app.rag.retriever import Retriever
from app.rag.pipeline import RAGPipeline

cfg = Settings(obs_enabled=False, embedding_dimension=128, rag_min_similarity=0.05, retrieval_top_k=4)
docs = load_knowledge_base(repo_root / "knowledge")
chunks = split_documents(docs, chunk_size=500, chunk_overlap=50)
emb = OfflineDeterministicEmbeddings(dimension=128)
vs = VectorStore.from_documents(chunks, emb.embed_documents(chunks), settings=cfg)
ret = Retriever(vectorstore=vs, embeddings=emb, settings=cfg)
pipe = RAGPipeline(retriever=ret, router=OfflineRuleSourceRouter(), generator=OfflineGroundedGenerator(), settings=cfg)

for scope in ("internal", "external", "all"):
    ans = pipe.run("Política de accesos y prevención OWASP", source_scope=scope)
    print(f"scope={scope} -> sources={[s.source_type for s in ans.sources]} | citations={ans.citations}")

# Caso de abstención cuando no hay evidencia suficiente
strict_cfg = Settings(obs_enabled=False, embedding_dimension=128, rag_min_similarity=0.99, retrieval_top_k=4)
strict_pipe = RAGPipeline(
    retriever=Retriever(vectorstore=vs, embeddings=emb, settings=strict_cfg),
    router=OfflineRuleSourceRouter(),
    generator=OfflineGroundedGenerator(),
    settings=strict_cfg,
)
abstained_ans = strict_pipe.run("Consulta fuera de dominio sin evidencia")
print(f"Abstención verificada: abstained={abstained_ans.abstained} | answer={abstained_ans.answer}")
